In [ ]:
from visualization.visualization.compile_dataset import compile_dataset, get_age
from visualization.visualization.helpers.big_vs_small import read_in_sizes, read_in_velocities

from visualization.visualization.helpers.file_manip import find_all_files
from visualization.visualization.seaborn_visualizations import binned_tet_over_time, variable_over_time
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from visualization.visualization import Visualization

In [ ]:
## graphing parameters

# list of included ages. compile dataset expects each age name to correspond to an age in this list. to exclude ages from analysis, exclude from this list.
age_list = ["10dpf", "17dpf", "18dpf", "19dpf", "21dpf", "24dpf"]

# possible shape of row values
shape_of_rows = ['5_5_5', '4_4_4', '3_3_3', '3_3']

# colours used for each age
colour_by_age = Visualization.defn_colours(age_list, 'viridis')

# golder where graph pngs should be saved
graph_folder = "graphs"

In [ ]:
dataset1_fns = find_all_files("/home/chamomile/Thyme_lab/data/jackie/", ext="csv", exclude=["arm_save"])

age_dict = dict()
for fp in dataset1_fns:
    age = get_age(fp, age_list)

    if len(age) > 0:
        age_dict[fp] = age

sorted_age_dict = dict(sorted(age_dict.items(), key=lambda age: int(age[1].strip('dpf'))))
d = read_in_sizes('comparecorrect/ymaze_age_size_jackie')
v_d = read_in_velocities('comparecorrect/ymaze_age_velocity_jackie')

In [ ]:
dataset2_fns = find_all_files("/home/chamomile/Thyme_lab/data/jinyue/", ext="csv", include="", exclude=["arm_save"])
age_dict = dict()
for fp in dataset2_fns:
    age = get_age(fp, age_list)

    if len(age) > 0:
        age_dict[fp] = age

sorted_age_dict_2 = dict(sorted(age_dict.items(), key=lambda age: int(age[1].strip('dpf'))))
d2 = read_in_sizes('comparecorrect/ymaze_age_size_jinyue')
v_d2 = read_in_velocities('comparecorrect/ymaze_age_velocity_jinyue')

In [ ]:
tetr_df1, age_totals1 = compile_dataset(sorted_age_dict, 30, age_list, shape_of_rows, velocity_dict=v_d)

In [ ]:
print(v_d2.keys())
tetr_df2, age_totals1 = compile_dataset(sorted_age_dict_2, 15, age_list, shape_of_rows, velocity_dict=v_d2)

In [ ]:
sns.set_context("talk", font_scale=1.8)
df1_colours = []
for age in tetr_df1['genotype'].unique():
    df1_colours.append(colour_by_age[age])

allgenos = tetr_df1['genotype']
tetr_df1_ = tetr_df1.assign(genotype=[f'{geno}, n={len(set(tetr_df1.loc[tetr_df1["genotype"] == geno,"global_id"].tolist()))}' for geno in allgenos])

variable_over_time(tetr_df1_, "Habituated Arm %", 'habitarm_binned', palette=df1_colours, legendin=False, save_fn=f"{graph_folder}/habitarm.png")

In [ ]:
variable_over_time(tetr_df1, "Total Tetragrams by Age", 'total_tetragrams', palette=df1_colours, save_fn=f"{graph_folder}/tottetr.png", legendin=False)

In [ ]:
# LRLR % over time for every age - shown on different graphs
for age in age_list:
    colour = colour_by_age[age]
    if len(tetr_df1[tetr_df1['genotype'] == age]) > 0:
        binned_tet_over_time(tetr_df1[tetr_df1['genotype'] == age], f"{age} Tetragram %", sep_by_age=True, palette=[colour], save_fn=f"{graph_folder}/{age}_tetragram_perc.png")


In [ ]:
twentyone_dpf = tetr_df1[tetr_df1['genotype'] == '21dpf']
twentyone_dpf.head()

In [ ]:
# 21 dpf - habitarm over time
variable_over_time(twentyone_dpf, "Habituated Arm % Over Time", 'habitarm_binned', palette=[colour_by_age['21dpf']], save_fn=f"{graph_folder}/habitarm_21dpf.png")
# 21 dpf - nanarm over time
variable_over_time(twentyone_dpf, "% of Time Spent in Center of Y", 'nanarm_binned', palette=[colour_by_age['21dpf']], save_fn=f"{graph_folder}/nanarm_21dpf.png")

In [ ]:
plt.figure(figsize=(12, 8))
sns.barplot(data=twentyone_dpf, x='time_bin', y='nanarm_binned', hue='genotype', estimator='mean', errorbar='se', palette=[colour_by_age['21dpf']])
plt.figure(figsize=(12, 8))
sns.barplot(data=twentyone_dpf, x='time_bin', y='habitarm_binned', hue='genotype', estimator='mean', errorbar='se', palette=[colour_by_age['21dpf']])

In [ ]:
plt.figure(figsize=(12, 8))
#ax = plt.subplot(121)
plt.ylim([0,100])
sns.barplot(data=tetr_df1_, y='habitarm_binned', hue='genotype', estimator='mean', errorbar='se', palette=df1_colours, legend=False)
#ax = plt.subplot(122)
plt.savefig('habitarm_total.png')
plt.figure(figsize=(12, 8))
plt.ylim([0,100])
sns.barplot(data=tetr_df1_, y='nanarm_binned', hue='genotype', estimator='mean', errorbar='se', palette=df1_colours, legend=False)
#plt.tight_layout()
plt.savefig('nanarm_total.png')

In [ ]:
# paired bar div. by age
plt.figure(figsize=(12, 8))
ax = plt.subplot(111)
sns.barplot(data=tetr_df1, x='time_bin', y='alternation_percent', hue='genotype', estimator='mean', errorbar='se', palette=df1_colours)
ax.legend(bbox_to_anchor=(1.01, 1))
plt.xticks([i for i in range(6)], [10, 20, 30, 40, 50, 60])

plt.savefig(f'{graph_folder}/age_alt_percent_bar_graph_binned.png',bbox_inches='tight')

In [ ]:
# top 20% vs low 20% (jackie)
for age in ['10dpf', '21dpf']:
    a = tetr_df1.loc[tetr_df1['genotype'] == age]

    z = np.array(a['size'])
    if len(z) > 0:
        m = np.median(z)
        med = np.quantile(z, [0.5])
        lowtwenty = a[z < med]
        toptwenty = a[z > med]
        # how often are they above chance versus comparing to any kind of when they're spiking?

        #        low, high = np.quantile(z, [0.25, 0.75])
        #        lowtwenty = a[z < low]
        #        toptwenty = a[z > high]
        
        lowtwenty = lowtwenty.assign(genotype=[f'{age}_low' for i in range(len(lowtwenty))])
        toptwenty = toptwenty.assign(genotype=[f'{age}_top' for i in range(len(toptwenty))])

        #title = f"Alternation Strategy Over Time for Mutants vs Wild Types, Shape of Ys = {shape_of_rows}, n = {len(set(lowtwenty['global_id'].tolist()))}"
        title = f"Alternation Strategy Over Time, n = {len(set(lowtwenty['global_id'].tolist()))}"
        #binned_tet_over_time(lowtwenty, title, palette=[colour_by_age[age]], save_fn=f"{age}_lowfifty.png")
        combined_df = pd.concat([lowtwenty, toptwenty])
        combined_df.to_csv(f'{age}highlow.csv')
        colours = [colour_by_age[age], colour_by_age[age]]
        variable_over_time(combined_df, title, 'alternation_percent', palette=colours, legendin=False, divbyline=True, save_fn=f"{graph_folder}/{age}_altperc_lowfifty.png")
        variable_over_time(combined_df, "Num of Tetragrams", 'total_tetragrams', palette=colours, legendin=False, divbyline=True, save_fn=f"{graph_folder}/{age}_numtet_lowfifty.png")
        variable_over_time(combined_df, "Habituated Arm %", 'habitarm_binned', palette=colours, legendin=False, divbyline=True, save_fn=f"{graph_folder}/{age}_habitarm_lowfifty.png")
